In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   5. Single Source File: hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
   
   BUSINESS QUESTION: 
   Number of the unit's active third party vendors who are single source vendors AND 
   located in high risk jurisdictions.
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU List + Mapping Bootstrap, strictly 
      filtered by 'jurisdiction' for Canada, Hong Kong, and Barbados.
   2. SINGLE SOURCE FILTER: Inner joins the main TP data to the published_contrcats 
      table via EngagementNumber to strictly keep single-source engagements.
   3. HIGH RISK EVALUATION: Checks three columns against the ABAC High Risk list. 
      If all three are blank/null, automatically defaults to High Risk.
   4. STRING MAPPING: Maps engagements to AUs by checking if the AU_Name exists 
      inside the text of the 'owning_lob' or 'lob_using' columns.
   5. FINAL OUTPUT: Strict 6-column structure, counting the distinct numerical 
      engagements and converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(business_segment) AS Business_Segment 
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has cost centers mapped to it
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    -- 4. Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- 5. Isolate the Engagement Numbers from the published single source file
    SELECT DISTINCT EngagementNumber
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE EngagementNumber IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 6. Extract TP data, INNER JOIN to single source list, and flag missing jurisdictions
    SELECT 
        p.EngagementNumber,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        -- Default to high risk if no jurisdiction is provided
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    -- STRICT FILTER: Only keep engagements that exist in the single source file
    INNER JOIN Single_Source_Engagements s
        ON p.EngagementNumber = s.EngagementNumber
),

High_Risk_Engagements AS (
    -- 7. Filter for engagements that touch a High Risk country or have NO country
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Engagements_By_AU AS (
    -- 8. Map high-risk single-source engagements to AUs via string matching
    SELECT 
        a.Base_AU_ID,
        COUNT(DISTINCT hr.EngagementNumber) AS High_Risk_Count
    FROM All_Base_AUs a
    INNER JOIN High_Risk_Engagements hr
        ON UPPER(hr.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
        OR UPPER(hr.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    GROUP BY a.Base_AU_ID
)

-- 9. Final Template: Strict 6-column output
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Base_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP04 - TP Sole Source Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   5. Single Source File: hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
   
   PURPOSE: 
   Provides a row-by-row view of every Third Party Engagement that triggered a count 
   for TP04. Confirms the engagement exists in the single source file and details 
   which specific country rule triggered the High Risk flag.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    SELECT DISTINCT EngagementNumber
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE EngagementNumber IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        p.EngagementNumber,
        p.ThirdPartyName,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    INNER JOIN Single_Source_Engagements s
        ON p.EngagementNumber = s.EngagementNumber
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    tp.EngagementNumber,
    tp.ThirdPartyName,
    'Yes' AS Verified_In_Single_Source_File,
    tp.owning_lob AS Raw_Col_K_owning_lob,
    tp.lob_using AS Raw_Col_L_lob_using,
    tp.location_of_tp AS Country_1,
    tp.country_prod_serv_originates AS Country_2,
    tp.Td_Countries_Impacted AS Country_3,
    
    CASE 
        WHEN tp.Is_Missing_Jurisdiction = 1 THEN 'Missing Jurisdiction (Auto High Risk)'
        ELSE 'Matched High Risk ABAC List' 
    END AS Risk_Flag_Reason
    
FROM All_Base_AUs a

-- Inner join via string matching
INNER JOIN Third_Party_Risk_Data tp
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    
LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName

-- Strictly filter for only the high-risk engagements
WHERE h1.CountryName IS NOT NULL
   OR h2.CountryName IS NOT NULL
   OR h3.CountryName IS NOT NULL
   OR tp.Is_Missing_Jurisdiction = 1
   
ORDER BY a.Base_AU_ID;